In [1]:
import pandas as pd
from google.colab import files

up = files.upload()                                   # pick features_targets_v2.csv
df = pd.read_csv(next(iter(up)), parse_dates=["month"], dtype={"account_id": str})
df = df.sort_values(["account_id", "month"]).reset_index(drop=True)

checks = {}
checks["rows"] = len(df)                                                        # expect 2100
checks["duplicate_account_months"] = int(df.duplicated(["account_id", "month"]).sum())   # expect 0

months = sorted(df["month"].unique())
checks["months"] = [str(m)[:7] for m in months]                                  # Mar -> Sep
per_account = df.groupby("account_id")["month"].nunique()
checks["accounts_missing_months"] = int((per_account != len(months)).sum())     # expect 0

prev_target = df.groupby("account_id")["target_balance"].shift(1)                # last month's answer
checks["lag1_not_equal_prev_target"] = int(((df["balance_lag1"] - prev_target).abs() > 0.01).sum())  # expect 0

missing = df.isna().sum()
checks["columns_with_missing"] = missing[missing > 0].to_dict()                  # only lag/std/avg columns
checks

Saving features_targets_v2.csv to features_targets_v2.csv


{'rows': 2100,
 'duplicate_account_months': 0,
 'months': ['2022-03',
  '2022-04',
  '2022-05',
  '2022-06',
  '2022-07',
  '2022-08',
  '2022-09'],
 'accounts_missing_months': 0,
 'lag1_not_equal_prev_target': 0,
 'columns_with_missing': {'balance_lag1': 300,
  'balance_lag2': 600,
  'balance_lag3': 900,
  'balance_mean_3m': 300,
  'balance_std_3m': 600,
  'balance_change': 600,
  'credit_lag1': 300,
  'debit_lag1': 300,
  'net_lag1': 300,
  'credit_avg_3m': 300,
  'debit_avg_3m': 300,
  'net_avg_3m': 300,
  'txn_count_lag1': 300,
  'merchant_count_lag1': 300,
  'spend_food_lag1': 300,
  'spend_travel_lag1': 300,
  'spend_shopping_lag1': 300,
  'spend_bills_lag1': 300,
  'spend_health_education_lag1': 300,
  'spend_cash_transfers_lag1': 300,
  'spend_other_lag1': 300}}

In [2]:
df.head()

,account_id,month,balance_lag1,balance_lag2,balance_lag3,balance_mean_3m,balance_std_3m,balance_change,credit_lag1,debit_lag1,...,spend_food_lag1,spend_travel_lag1,spend_shopping_lag1,spend_bills_lag1,spend_health_education_lag1,spend_cash_transfers_lag1,spend_other_lag1,month_of_year,days_in_month,target_balance
0,501893368,2022-03-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,31,0.00
1,501893368,2022-04-01,0.00,NaN,NaN,0.000000,NaN,NaN,0.0,0.00,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,4,30,0.00
2,501893368,2022-05-01,0.00,0.00,NaN,0.000000,0.000000,0.00,0.0,0.00,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,5,31,-42.55
3,501893368,2022-06-01,-42.55,0.00,0.0,-14.183333,24.566254,-42.55,0.0,42.55,...,0.0,0.0,0.0,42.55,0.0,0.0,0.0,6,30,-42.55
4,501893368,2022-07-01,-42.55,-42.55,0.0,-28.366667,24.566254,0.00,0.0,0.00,...,0.0,0.0,0.0,0.00,0.0,0.0,0.0,7,31,-443.75


In [3]:

df = df[df["month"] > df["month"].min()].reset_index(drop=True)

months = sorted(df["month"].unique())
n_test = max(1, round(len(months) * 0.2))
TRAIN_MONTHS = months[:-n_test]
TEST_MONTHS  = months[-n_test:]
FOLD_MONTHS  = TRAIN_MONTHS[-3:]

train = df[df["month"].isin(TRAIN_MONTHS)]
test  = df[df["month"].isin(TEST_MONTHS)]
assert train["month"].max() < test["month"].min()

print("train:", [str(m)[:7] for m in TRAIN_MONTHS], len(train), "rows")
print("test :", [str(m)[:7] for m in TEST_MONTHS], len(test), "rows")
print("folds:", [str(m)[:7] for m in FOLD_MONTHS])

train: ['2022-04', '2022-05', '2022-06', '2022-07', '2022-08'] 1500 rows
test : ['2022-09'] 300 rows
folds: ['2022-06', '2022-07', '2022-08']


In [4]:
ID_COLS, TARGET = ["account_id", "month"], "target_balance"
feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET]]

def useless_columns(part, cols):
    """Constant columns and exact copies, looking ONLY at the rows given (the training months)."""
    drop = {c: "constant" for c in cols if part[c].nunique(dropna=True) <= 1}
    rest = [c for c in cols if c not in drop]
    corr = part[rest].corr().abs()
    for i, a in enumerate(rest):
        for b in rest[i + 1:]:
            if a in drop or b in drop or corr.loc[a, b] < 0.999:
                continue

            worse = b if part[b].isna().sum() >= part[a].isna().sum() else a
            drop[worse] = f"copy of {b if worse == a else a}"
    return drop

DROPPED = useless_columns(train, feature_cols)
FEATURES = [c for c in feature_cols if c not in DROPPED]
print(len(FEATURES), "features kept")
DROPPED

20 features kept


{'n_accounts': 'constant',
 'spend_other_lag1': 'constant',
 'balance_change': 'copy of net_lag1',
 'merchant_count_lag1': 'copy of txn_count_lag1'}

In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

def move(p):                      # what the models learn: the change this month
    return p[TARGET] - p["balance_lag1"]

def folds():                      # walk-forward inside the training months
    for m in FOLD_MONTHS:
        yield train[train["month"] < m], train[train["month"] == m]

def score(p, pred):
    y, lag1 = p[TARGET], p["balance_lag1"]
    return {"MAE": mean_absolute_error(y, pred), "R2_change": r2_score(y - lag1, pred - lag1)}

# ---- the models: fit(train part) -> a function that predicts balances ----
def fit_naive(tr):  return lambda p: p["balance_lag1"]
def fit_3m(tr):     return lambda p: p["balance_mean_3m"]
def fit_drift(tr):
    m = move(tr).median()                                  # median move of all accounts, PAST months only
    return lambda p: p["balance_lag1"] + m

LIN_FEATURES = [c for c in FEATURES if c not in ["month_of_year", "debit_lag1", "debit_avg_3m"]]
def fit_linear(tr, make_model):
    na_cols = [c for c in LIN_FEATURES if tr[c].isna().any()]   # decided on train only
    def X(p):
        x = p[LIN_FEATURES].copy()
        for c in na_cols:
            x[c + "_missing"] = x[c].isna().astype(int)         # one 0/1 flag per filled column
        return x.fillna(0)
    scaler = StandardScaler().fit(X(tr))
    model = make_model().fit(scaler.transform(X(tr)), move(tr))
    return lambda p: p["balance_lag1"] + model.predict(scaler.transform(X(p)))

def fit_boost(tr):
    model = HistGradientBoostingRegressor(loss="absolute_error", max_depth=3, learning_rate=0.05,
                                          max_iter=200, min_samples_leaf=20, random_state=0)
    model.fit(tr[FEATURES], move(tr))                       # NaN left as NaN (her rule)
    return lambda p: p["balance_lag1"] + model.predict(p[FEATURES])

MODELS = {
    "naive": fit_naive,
    "drift": fit_drift,
    "3-month average": fit_3m,
    "linear": lambda tr: fit_linear(tr, LinearRegression),
    "median linear": lambda tr: fit_linear(tr, lambda: QuantileRegressor(quantile=0.5, alpha=0, solver="highs")),
    "boosting (his settings)": fit_boost,
}

rows = []
for name, fit in MODELS.items():
    for tr, va in folds():
        predict = fit(tr)
        rows.append({"model": name, "month": str(va["month"].iloc[0])[:7],
                     "train_MAE": mean_absolute_error(tr[TARGET], predict(tr)), **score(va, predict(va))})
comparison = pd.DataFrame(rows)

print(comparison.pivot(index="month", columns="model", values="MAE").round(1), "\n")
pooled = comparison.groupby("model")[["train_MAE", "MAE", "R2_change"]].mean()
pooled["skill_vs_naive"] = 1 - pooled["MAE"] / pooled.loc["naive", "MAE"]
pooled["skill_vs_drift"] = 1 - pooled["MAE"] / pooled.loc["drift", "MAE"]
pooled.round(3).sort_values("MAE")

model    3-month average  boosting (his settings)  drift  linear  \
month                                                              
2022-06            626.3                    306.4  310.1   389.2   
2022-07            590.8                    277.1  275.6   444.2   
2022-08            755.5                    463.8  467.4   469.2   

model    median linear  naive  
month                          
2022-06          311.9  317.5  
2022-07          283.2  278.1  
2022-08          472.1  481.6   



,train_MAE,MAE,R2_change,skill_vs_naive,skill_vs_drift
model,,,,,
boosting (his settings),293.132,349.097,-0.138,0.028,0.006
drift,316.361,351.031,-0.183,0.022,0.000
median linear,312.918,355.741,-0.177,0.009,-0.013
naive,320.764,359.042,-0.251,0.000,-0.023
linear,372.071,434.186,-0.169,-0.209,-0.237
3-month average,458.486,657.553,-1.620,-0.831,-0.873


In [6]:
up = files.upload()
tx = pd.read_csv(next(iter(up)),
                 usecols=["account_id", "txn_date_time", "billing_amount", "txn_type", "mcc_code"],
                 parse_dates=["txn_date_time"], dtype={"account_id": str, "mcc_code": str})
print(tx.shape)                                        # expect (2296, 5)
print(tx["txn_date_time"].min(), "->", tx["txn_date_time"].max())
tx.head()

Saving transactions_cleaned_202611061519.csv to transactions_cleaned_202611061519.csv
(2296, 5)
2022-03-01 00:19:00 -> 2022-09-30 18:52:02


,account_id,txn_date_time,billing_amount,txn_type,mcc_code
0,501893368,2022-05-15 01:58:42,-42.55,Settled,7997
1,501893368,2022-07-06 14:26:29,-401.20,Settled,5999
2,501893368,2022-09-06 16:55:00,-459.35,Settled,4900
3,501893368,2022-09-14 06:22:00,-116.65,Settled,7311
4,501893368,2022-09-21 16:16:00,-37.03,Settled,5814


In [7]:
def spending_speed(tx, month_start):
    """Group 1 for ONE month. Step 1: drop every transaction on/after the cutoff."""
    past = tx[tx["txn_date_time"] < month_start]
    spend = past[past["billing_amount"] < 0]                       # money out only
    def window(days):
        w = spend[spend["txn_date_time"] >= month_start - pd.Timedelta(days=days)]
        return -w.groupby("account_id")["billing_amount"].sum()
    return pd.DataFrame({"spend_last_7d": window(7), "spend_last_30d": window(30)})

parts = []
for m in df["month"].unique():
    f = spending_speed(tx, m).reset_index()
    f["month"] = m
    parts.append(f)
speed = pd.concat(parts)

G1 = ["spend_last_7d", "spend_last_30d", "burn_accel"]
df = df.drop(columns=G1, errors="ignore").merge(speed, on=["account_id", "month"], how="left")
df[["spend_last_7d", "spend_last_30d"]] = df[["spend_last_7d", "spend_last_30d"]].fillna(0)   # no spending = a real 0

rate7, rate30 = df["spend_last_7d"] / 7, df["spend_last_30d"] / 30
df["burn_accel"] = np.minimum(rate7, 3 * rate30) - rate30

# rebuild train / test with the SAME months as before (don't re-run Cell 1: it would drop April!)
train = df[df["month"].isin(TRAIN_MONTHS)]
test  = df[df["month"].isin(TEST_MONTHS)]
print(train.shape, test.shape)

(1500, 30) (300, 30)


In [8]:
train.head()

,account_id,month,balance_lag1,balance_lag2,balance_lag3,balance_mean_3m,balance_std_3m,balance_change,credit_lag1,debit_lag1,...,spend_bills_lag1,spend_health_education_lag1,spend_cash_transfers_lag1,spend_other_lag1,month_of_year,days_in_month,target_balance,spend_last_7d,spend_last_30d,burn_accel
0,501893368,2022-04-01,0.00,NaN,NaN,0.000000,NaN,NaN,0.0,0.00,...,0.00,0.0,0.0,0.0,4,30,0.00,0.0,0.00,0.000000
1,501893368,2022-05-01,0.00,0.00,NaN,0.000000,0.000000,0.00,0.0,0.00,...,0.00,0.0,0.0,0.0,5,31,-42.55,0.0,0.00,0.000000
2,501893368,2022-06-01,-42.55,0.00,0.00,-14.183333,24.566254,-42.55,0.0,42.55,...,42.55,0.0,0.0,0.0,6,30,-42.55,0.0,42.55,-1.418333
3,501893368,2022-07-01,-42.55,-42.55,0.00,-28.366667,24.566254,0.00,0.0,0.00,...,0.00,0.0,0.0,0.0,7,31,-443.75,0.0,0.00,0.000000
4,501893368,2022-08-01,-443.75,-42.55,-42.55,-176.283333,231.632928,-401.20,0.0,401.20,...,0.00,0.0,0.0,0.0,8,31,-443.75,0.0,401.20,-13.373333


In [9]:
print((train[G1] != 0).mean().round(3))                                   # share of non-zero rows
print(train[G1 + ["debit_lag1"]].corr().round(3)["debit_lag1"])           # copy of an existing column?

spend_last_7d     0.222
spend_last_30d    0.617
burn_accel        0.617
dtype: float64
spend_last_7d     0.527
spend_last_30d    0.998
burn_accel       -0.052
debit_lag1        1.000
Name: debit_lag1, dtype: float64


In [10]:
G1_USE = ["spend_last_7d", "burn_accel"]

def fit_boost_cols(cols):
    """The same boosting (your settings), but with a chosen list of columns."""
    def fit(tr):
        model = HistGradientBoostingRegressor(loss="absolute_error", max_depth=3, learning_rate=0.05,
                                              max_iter=200, min_samples_leaf=20, random_state=0)
        model.fit(tr[cols], move(tr))
        return lambda p: p["balance_lag1"] + model.predict(p[cols])
    return fit

def fold_errors(fit):
    """The miss of every validation row, fold by fold (3 x 300 rows)."""
    return [(va[TARGET] - fit(tr)(va)).abs().to_numpy() for tr, va in folds()]

def compare(fit_without, fit_with, n_boot=1000, seed=0, show=True):
    e0, e1 = fold_errors(fit_without), fold_errors(fit_with)
    diff = np.concatenate(e1) - np.concatenate(e0)
    rng = np.random.default_rng(seed)
    lo, hi = np.percentile([rng.choice(diff, len(diff)).mean() for _ in range(n_boot)], [2.5, 97.5])
    if show:
        for m, a, b in zip(FOLD_MONTHS, e0, e1):
            print(str(m)[:7], " without:", round(a.mean(), 1), " with:", round(b.mean(), 1))
        print("pooled MAE change (with - without):", round(diff.mean(), 2), "| 95%:", round(lo, 2), "to", round(hi, 2))
    return diff.mean(), lo, hi

compare(fit_boost_cols(FEATURES), fit_boost_cols(FEATURES + G1_USE))

2022-06  without: 306.4  with: 306.9
2022-07  without: 277.1  with: 277.1
2022-08  without: 463.8  with: 464.6
pooled MAE change (with - without): 0.42 | 95%: -1.74 to 2.58


(np.float64(0.42346671464243907),
 np.float64(-1.7350692076503949),
 np.float64(2.576748992149187))

In [11]:
ABLATION = []   # every keep/drop decision, with its numbers
ABLATION.append({"group": "G1 spending speed (spend_last_7d, burn_accel)",
                 "mae_change": 0.42, "ci_low": -1.74, "ci_high": 2.58, "decision": "drop"})

In [12]:
df = df.sort_values(["account_id", "month"]).reset_index(drop=True)

# 1. the account's own median move over its last 3 months
df["move_median_3m"] = (df.groupby("account_id")["net_lag1"]
                          .transform(lambda s: s.rolling(3, min_periods=1).median()))

# 2. the median move of ALL accounts in all months before M
med = {m: df.loc[df["month"] <= m, "net_lag1"].median() for m in sorted(df["month"].unique())}
df["all_accounts_median_move"] = df["month"].map(med)

G2 = ["move_median_3m", "all_accounts_median_move"]
train = df[df["month"].isin(TRAIN_MONTHS)]
test  = df[df["month"].isin(TEST_MONTHS)]

print((train[G2] != 0).mean().round(3))                                   # share of non-zero rows
print(train[G2 + ["net_lag1", "net_avg_3m"]].corr().round(3)[G2])         # copies of existing columns?
print(df.groupby("month")["all_accounts_median_move"].first().round(1))   # one value per month

move_median_3m              0.706
all_accounts_median_move    1.000
dtype: float64
                          move_median_3m  all_accounts_median_move
move_median_3m                     1.000                     0.173
all_accounts_median_move           0.173                     1.000
net_lag1                           0.645                     0.021
net_avg_3m                         0.881                     0.033
month
2022-04-01   -91.8
2022-05-01   -81.1
2022-06-01   -55.1
2022-07-01   -71.6
2022-08-01   -57.6
2022-09-01   -76.4
Name: all_accounts_median_move, dtype: float64


In [13]:
compare(fit_boost_cols(FEATURES), fit_boost_cols(FEATURES + G2))

2022-06  without: 306.4  with: 306.4
2022-07  without: 277.1  with: 277.8
2022-08  without: 463.8  with: 462.2
pooled MAE change (with - without): -0.32 | 95%: -1.27 to 0.58


(np.float64(-0.31712704926512236),
 np.float64(-1.2679653654914909),
 np.float64(0.5842992088188648))

In [14]:
ABLATION.append({"group": "G2 past moves (move_median_3m, all_accounts_median_move)",
                 "mae_change": -0.32, "ci_low": -1.27, "ci_high": 0.58, "decision": "drop"})

In [15]:
def lumpiness(tx, month_start):
    past = tx[(tx["txn_date_time"] < month_start) &                                  # cutoff
              (tx["txn_date_time"] >= month_start - pd.DateOffset(months=3))]        # last 3 months
    pay = past[past["billing_amount"] < 0].assign(amt=lambda d: -d["billing_amount"])
    g = pay.groupby("account_id")["amt"]
    out = pd.DataFrame({"max_debit_3m": g.max(), "total_3m": g.sum(), "median_pay": g.median()})
    big = pay.merge(out["median_pay"], left_on="account_id", right_index=True)
    out["n_big_3m"] = (big["amt"] > 2 * big["median_pay"]).groupby(big["account_id"]).sum()
    return out

parts = []
for m in df["month"].unique():
    f = lumpiness(tx, m).reset_index()
    f["month"] = m
    parts.append(f)
lump = pd.concat(parts)

G3 = ["max_debit_3m", "lump_ratio", "n_big_3m"]
df = df.drop(columns=G3 + ["total_3m", "median_pay"], errors="ignore").merge(
        lump, on=["account_id", "month"], how="left")
df["lump_ratio"] = df["max_debit_3m"] / (df["total_3m"] / 3)       # NaN when nothing was spent
df = df.drop(columns=["total_3m", "median_pay"])

train = df[df["month"].isin(TRAIN_MONTHS)]
test  = df[df["month"].isin(TEST_MONTHS)]
print(train[G3].notna().mean().round(3))                                         # share with a value
print(train[G3 + ["debit_avg_3m", "debit_lag1"]].corr().round(3)[G3])           # copies?

max_debit_3m    0.871
lump_ratio      0.871
n_big_3m        0.871
dtype: float64
              max_debit_3m  lump_ratio  n_big_3m
max_debit_3m         1.000       0.048     0.405
lump_ratio           0.048       1.000    -0.289
n_big_3m             0.405      -0.289     1.000
debit_avg_3m         0.853      -0.100     0.365
debit_lag1           0.584      -0.063     0.251


In [16]:
compare(fit_boost_cols(FEATURES), fit_boost_cols(FEATURES + G3))

2022-06  without: 306.4  with: 306.4
2022-07  without: 277.1  with: 278.2
2022-08  without: 463.8  with: 461.3
pooled MAE change (with - without): -0.47 | 95%: -3.02 to 1.93


(np.float64(-0.47094972273144914),
 np.float64(-3.0246275895426575),
 np.float64(1.9290897731091856))

In [17]:
ABLATION.append({"group": "G3 lumpiness (max_debit_3m, lump_ratio, n_big_3m)",
                 "mae_change": -0.47, "ci_low": -3.02, "ci_high": 1.93, "decision": "drop"})

In [18]:
GROUPS = {
    "balance":  ["balance_lag1", "balance_lag2", "balance_lag3", "balance_mean_3m", "balance_std_3m"],
    "money flow": ["credit_lag1", "debit_lag1", "net_lag1", "credit_avg_3m", "debit_avg_3m", "net_avg_3m"],
    "activity": ["txn_count_lag1"],
    "category spend": [c for c in FEATURES if c.startswith("spend_")],
    "calendar": ["month_of_year", "days_in_month"],
}
rows = []
for name, cols in GROUPS.items():
    without = [c for c in FEATURES if c not in cols]
    change, lo, hi = compare(fit_boost_cols(without), fit_boost_cols(FEATURES), show=False)
    rows.append({"group": name, "mae_change_when_kept": round(change, 2),
                 "ci_low": round(lo, 2), "ci_high": round(hi, 2)})
pd.DataFrame(rows)

,group,mae_change_when_kept,ci_low,ci_high
0,balance,-0.64,-3.71,2.57
1,money flow,-2.23,-5.42,1.08
2,activity,-0.95,-2.42,0.54
3,category spend,-0.66,-3.46,2.01
4,calendar,-0.33,-1.87,1.40


In [19]:
SMALL = [c for c in FEATURES if c not in GROUPS["calendar"] + GROUPS["category spend"]]
print(len(SMALL), "features:", SMALL)
compare(fit_boost_cols(FEATURES), fit_boost_cols(SMALL))

12 features: ['balance_lag1', 'balance_lag2', 'balance_lag3', 'balance_mean_3m', 'balance_std_3m', 'credit_lag1', 'debit_lag1', 'net_lag1', 'credit_avg_3m', 'debit_avg_3m', 'net_avg_3m', 'txn_count_lag1']
2022-06  without: 306.4  with: 308.0
2022-07  without: 277.1  with: 280.5
2022-08  without: 463.8  with: 461.1
pooled MAE change (with - without): 0.76 | 95%: -2.04 to 3.76


(np.float64(0.7618087381072639),
 np.float64(-2.0394191762884684),
 np.float64(3.7563572037335384))

In [20]:
FEATURES_FINAL = SMALL
ABLATION.append({"group": "calendar + category spend (8 cols)", "mae_change": 0.76,
                 "ci_low": -2.04, "ci_high": 3.76, "decision": "drop (tie -> keep smaller set)"})
pd.DataFrame(ABLATION)

,group,mae_change,ci_low,ci_high,decision
0,"G1 spending speed (spend_last_7d, burn_accel)",0.42,-1.74,2.58,drop
1,"G2 past moves (move_median_3m, all_accounts_me...",-0.32,-1.27,0.58,drop
2,"G3 lumpiness (max_debit_3m, lump_ratio, n_big_3m)",-0.47,-3.02,1.93,drop
3,calendar + category spend (8 cols),0.76,-2.04,3.76,drop (tie -> keep smaller set)


In [21]:
!pip install -q optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def fit_hgb(cols, params):
    """Boosting with chosen columns AND chosen settings."""
    def fit(tr):
        model = HistGradientBoostingRegressor(loss="absolute_error", random_state=0, **params)
        model.fit(tr[cols], move(tr))
        return lambda p: p["balance_lag1"] + model.predict(p[cols])
    return fit

def cv_mae(fit):
    return np.concatenate(fold_errors(fit)).mean()          # pooled MAE over Jun, Jul, Aug

def objective(trial):
    params = {
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_iter":          trial.suggest_int("max_iter", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 2, 6),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 10, 80),
        "l2_regularization": trial.suggest_float("l2_regularization", 0.001, 10, log=True),
    }
    return cv_mae(fit_hgb(FEATURES_FINAL, params))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=50, timeout=600)

DEFAULT = {"learning_rate": 0.05, "max_iter": 200, "max_depth": 3, "min_samples_leaf": 20}
print("his settings :", round(cv_mae(fit_hgb(FEATURES_FINAL, DEFAULT)), 2))
print("best tuned   :", round(study.best_value, 2), study.best_params)

his settings : 349.86
best tuned   : 346.89 {'learning_rate': 0.01427379109280977, 'max_iter': 252, 'max_depth': 6, 'min_samples_leaf': 41, 'l2_regularization': 3.6952247972031786}


In [22]:
compare(fit_hgb(FEATURES_FINAL, DEFAULT), fit_hgb(FEATURES_FINAL, study.best_params))

2022-06  without: 308.0  with: 305.4
2022-07  without: 280.5  with: 273.1
2022-08  without: 461.1  with: 462.1
pooled MAE change (with - without): -2.97 | 95%: -6.48 to 0.56


(np.float64(-2.9687898428165957),
 np.float64(-6.483027222820524),
 np.float64(0.5624405410013346))

In [23]:
print("drift vs your settings:")
compare(fit_drift, fit_hgb(FEATURES_FINAL, DEFAULT))
print("\ndrift vs tuned:")
compare(fit_drift, fit_hgb(FEATURES_FINAL, study.best_params))

drift vs your settings:
2022-06  without: 310.1  with: 308.0
2022-07  without: 275.6  with: 280.5
2022-08  without: 467.4  with: 461.1
pooled MAE change (with - without): -1.17 | 95%: -6.49 to 4.1

drift vs tuned:
2022-06  without: 310.1  with: 305.4
2022-07  without: 275.6  with: 273.1
2022-08  without: 467.4  with: 462.1
pooled MAE change (with - without): -4.14 | 95%: -8.35 to 0.33


(np.float64(-4.141380897391621),
 np.float64(-8.353976786378917),
 np.float64(0.3268673493243341))

In [24]:
def metrics(p, pred, pred_naive, pred_drift):
    y, lag1 = p[TARGET], p["balance_lag1"]
    err = (y - pred).abs()
    return {"MAE": err.mean(),
            "skill_vs_naive": 1 - err.mean() / (y - pred_naive).abs().mean(),
            "skill_vs_drift": 1 - err.mean() / (y - pred_drift).abs().mean(),
            "R2_change": r2_score(y - lag1, pred - lag1),
            "median_AE": err.median(),                 # typical miss, less hit by huge shocks
            "bias": (y - pred).mean(),                 # > 0: we guess too low, < 0: too high
            "within_100": (err <= 100).mean(),         # share of accounts missed by <= $100
            "within_500": (err <= 500).mean()}

def inner_errors(hist, val_months, fit):
    """Walk-forward INSIDE the history only."""
    return np.concatenate([(hist[hist["month"] == v][TARGET]
                            - fit(hist[hist["month"] < v])(hist[hist["month"] == v])).abs().to_numpy()
                           for v in val_months])

def choose(hist, val_months, n_trials=20):
    """Pick the feature set, then tune with Optuna, using ONLY the history."""
    # 1. features: all vs the smaller set (keep the smaller one unless it's > $2 worse)
    e_all = inner_errors(hist, val_months, fit_hgb(FEATURES, DEFAULT)).mean()
    e_small = inner_errors(hist, val_months, fit_hgb(FEATURES_FINAL, DEFAULT)).mean()
    cols = FEATURES_FINAL if e_small <= e_all + 2 else FEATURES
    # 2. settings: Optuna on the inner folds
    def objective(trial):
        params = {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                  "max_iter": trial.suggest_int("max_iter", 100, 600),
                  "max_depth": trial.suggest_int("max_depth", 2, 6),
                  "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 80),
                  "l2_regularization": trial.suggest_float("l2_regularization", 0.001, 10, log=True)}
        return inner_errors(hist, val_months, fit_hgb(cols, params)).mean()
    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
    st.optimize(objective, n_trials=n_trials)
    # keep the default settings unless tuning wins by >= $2 (her rule)
    e_default = inner_errors(hist, val_months, fit_hgb(cols, DEFAULT)).mean()
    params = st.best_params if st.best_value <= e_default - 2 else DEFAULT
    return cols, params

In [25]:
EVAL_MONTHS = months[2:]                       # Jun, Jul, Aug, Sep (each has >= 2 months before it)
results, diffs = [], []
for m in EVAL_MONTHS:
    hist, now = df[df["month"] < m], df[df["month"] == m]
    val_months = [v for v in EVAL_MONTHS if v < m]          # inner folds: only months before m
    if val_months:
        cols, params = choose(hist, val_months)
    else:
        cols, params = FEATURES, DEFAULT                     # June: no history to choose with
    pred = fit_hgb(cols, params)(hist)(now)
    pred_naive = now["balance_lag1"]
    pred_drift = fit_drift(hist)(now)
    r = metrics(now, pred, pred_naive, pred_drift)
    r.update({"month": str(m)[:7], "n_features": len(cols), "tuned": params is not DEFAULT})
    results.append(r)
    diffs.append((now[TARGET] - pred).abs().to_numpy() - (now[TARGET] - pred_drift).abs().to_numpy())

nested = pd.DataFrame(results).set_index("month").round(3)
print(nested)

d = np.concatenate(diffs)
rng = np.random.default_rng(0)
lo, hi = np.percentile([rng.choice(d, len(d)).mean() for _ in range(1000)], [2.5, 97.5])
print("\nmodel vs drift, Jun-Sep pooled: MAE change", round(d.mean(), 2), "| 95%:", round(lo, 2), "to", round(hi, 2))

             MAE  skill_vs_naive  skill_vs_drift  R2_change  median_AE  \
month                                                                    
2022-06  306.363           0.035           0.012     -0.160    102.721   
2022-07  283.563          -0.020          -0.029     -0.048    146.534   
2022-08  463.835           0.037           0.008     -0.189    153.058   
2022-09  385.867           0.006          -0.008     -0.101    146.371   

            bias  within_100  within_500  n_features  tuned  
month                                                        
2022-06 -231.865       0.480       0.857          20  False  
2022-07 -142.731       0.383       0.880          12   True  
2022-08 -392.258       0.397       0.777          20  False  
2022-09 -240.508       0.367       0.853          12  False  

model vs drift, Jun-Sep pooled: MAE change 0.96 | 95%: -3.78 to 5.84


In [26]:
BEST_PARAMS = study.best_params
def fit_shrunk(cols, params, w):
    """final move = w x boosting's move + (1 - w) x drift's move"""
    def fit(tr):
        model = HistGradientBoostingRegressor(loss="absolute_error", random_state=0, **params)
        model.fit(tr[cols], move(tr))
        med = move(tr).median()                                   # drift's move, from the PAST only
        return lambda p: p["balance_lag1"] + w * model.predict(p[cols]) + (1 - w) * med
    return fit

W_GRID = np.round(np.arange(0, 1.01, 0.1), 1)                    # 0.0, 0.1, ... 1.0
shrink = {w: cv_mae(fit_shrunk(FEATURES_FINAL, BEST_PARAMS, w)) for w in W_GRID}
print(pd.Series(shrink).round(2))
BEST_W = min(shrink, key=shrink.get)
print("best w:", BEST_W)

0.0    351.03
0.1    350.32
0.2    349.71
0.3    349.15
0.4    348.65
0.5    348.19
0.6    347.80
0.7    347.50
0.8    347.27
0.9    347.05
1.0    346.89
dtype: float64
best w: 1.0


In [27]:
predict = fit_hgb(FEATURES_FINAL, BEST_PARAMS)(train)
ev = test[["account_id", "balance_lag1", TARGET, "debit_avg_3m"]].copy()
ev["pred"] = predict(test)
ev["pred_drift"] = fit_drift(train)(test)
ev["real_move"] = ev[TARGET] - ev["balance_lag1"]
ev["err_model"] = (ev[TARGET] - ev["pred"]).abs()
ev["err_drift"] = (ev[TARGET] - ev["pred_drift"]).abs()
ev["err_naive"] = ev["real_move"].abs()

# 3 ways to split the accounts
ev["month_kind"] = np.where(ev["real_move"] == 0, "quiet",
                   np.where(ev["real_move"] > 0, "refund (up)", "spending"))
ev["spender"] = pd.qcut(ev["debit_avg_3m"].rank(method="first"), 3, labels=["low", "medium", "high"])
share = (df[df["month"] <= TEST_MONTHS[0]].groupby("account_id")["txn_count_lag1"]
           .apply(lambda s: (s > 0).mean()))                       # % of past months with activity
ev["user"] = np.where(ev["account_id"].map(share) >= 0.67, "active", "regular")

for col in ["month_kind", "spender", "user"]:
    t = ev.groupby(col, observed=True)[["err_model", "err_drift", "err_naive"]].mean().round(1)
    t.insert(0, "rows", ev.groupby(col, observed=True).size())
    print(t, "\n")

             rows  err_model  err_drift  err_naive
month_kind                                        
quiet         112      121.1       72.7        0.0
refund (up)    10      654.7      584.6      512.0
spending      178      534.9      566.4      625.7 

         rows  err_model  err_drift  err_naive
spender                                       
low       100      355.8      351.9      360.2
medium    100      433.5      424.5      423.1
high      100      363.9      371.7      381.6 

         rows  err_model  err_drift  err_naive
user                                          
active     99      385.4      377.6      383.1
regular   201      383.9      385.2      390.9 



In [28]:
start = TEST_MONTHS[0]
sep_tx = tx[(tx["txn_date_time"] >= start) & (tx["txn_date_time"] < start + pd.DateOffset(months=1))]
biggest = (sep_tx.assign(size=sep_tx["billing_amount"].abs())
                 .sort_values("size").groupby("account_id").tail(1)
                 .set_index("account_id")["billing_amount"])
worst = ev.nlargest(10, "err_model")[["account_id", "balance_lag1", TARGET, "pred", "err_model"]]
worst["biggest_sep_payment"] = worst["account_id"].map(biggest)
worst.round(1)

,account_id,balance_lag1,target_balance,pred,err_model,biggest_sep_payment
1595,923749577,-2024.7,-7043.5,-2213.7,4829.8,-3988.5
1013,762627724,-1181.6,-5407.1,-1402.6,4004.5,-3799.4
881,726986534,-1408.9,2365.3,-1626.6,3991.9,3774.2
587,659181829,-1837.8,-5764.4,-1868.3,3896.1,-3710.9
647,680525098,36.4,-3760.2,-13.8,3746.4,-3781.6
1673,954554020,-2614.5,-6439.2,-2698.8,3740.5,-3610.2
941,741201159,-1442.0,-5264.8,-1533.4,3731.4,-3708.2
1319,840736425,-2497.7,-6225.7,-2542.4,3683.3,-3377.8
47,511199434,-5170.8,-8970.2,-5505.8,3464.4,-3799.4
1409,859821899,-1363.0,-4624.7,-1546.1,3078.6,-2810.8
